# 🎓 Day 1 · Session 3
# 03C · Structured Outputs & Prompt Patterns
## JSON, markdown tables and reusable faculty prompt templates

## 🎯 Learning Objectives

- Generate structured outputs
- Use JSON mode
- Build reusable prompt patterns

## 🔧 Setup

Expected `.env` file in the same folder as this notebook:

```env
OPENAI_API_KEY=sk-your-key-here
```

In [ ]:
# Uncomment if required
# %pip install openai python-dotenv pandas

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv, dotenv_values
from openai import OpenAI
import pandas as pd
import json

In [ ]:
env_path = Path.cwd() / ".env"
print("Current working directory:", Path.cwd())
print("Looking for .env at:", env_path)
print("Env exists:", env_path.exists())

load_dotenv(env_path, override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("Dotenv keys found:", list(dotenv_values(env_path).keys()))
    raise ValueError("OPENAI_API_KEY not found. Please check your .env file.")

client = OpenAI(api_key=api_key)
print("OpenAI client initialized successfully.")

In [ ]:
def ask_openai(prompt, model="gpt-4o-mini", temperature=0.3,
               system_prompt="You are a helpful AI teaching assistant.",
               max_tokens=800):
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

# 1️⃣ Why Structured Outputs Matter

Free text is for humans. JSON/tables are for software.

In [ ]:
student_text = "Priya Sharma, Roll 21CSB044, scored 87 in ML and 92 in DSA."
prompt = f"Extract student information from this text:\n{student_text}"
print(ask_openai(prompt))

# 2️⃣ Ask for JSON

In [ ]:
json_prompt = f"""
Extract student information as valid JSON.

Required schema:
{{
  "name": "...",
  "roll_number": "...",
  "scores": {{
    "subject": marks
  }}
}}

Text:
{student_text}
"""
print(ask_openai(json_prompt))

# 3️⃣ JSON Mode

In [ ]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    temperature=0.2,
    response_format={"type": "json_object"},
    messages=[
        {"role": "system", "content": "Return valid JSON only."},
        {"role": "user", "content": json_prompt}
    ]
)
json_text = response.choices[0].message.content
print(json_text)
data = json.loads(json_text)
data

# 4️⃣ Markdown Table Output

In [ ]:
table_prompt = """
Compare zero-shot, few-shot and chain-of-thought prompting.

Return as a markdown table with columns:
Technique, Best Used For, Example Use Case, Risk.
"""
print(ask_openai(table_prompt))

# 5️⃣ Prompt Pattern: The Explainer

In [ ]:
explainer_template = """
You are a teacher for {level} students studying {subject}.

Explain {concept} using:
1. A one-sentence definition
2. A real-world analogy from {domain}
3. One worked example
4. One common misconception to avoid
5. One check-your-understanding question

Keep it under {word_limit} words.
"""
prompt = explainer_template.format(
    level="second-year engineering",
    subject="machine learning",
    concept="gradient descent",
    domain="hill climbing",
    word_limit=180
)
print(ask_openai(prompt))

# 6️⃣ Prompt Pattern: The Examiner

In [ ]:
examiner_prompt = """
Create 5 MCQ questions on supervised learning for second-year engineering students.

Difficulty: Medium

For each question include:
- Question
- Four options A-D
- Correct answer
- One-line explanation

Avoid repeated question patterns.
"""
print(ask_openai(examiner_prompt, max_tokens=1200))

# 7️⃣ Prompt Pattern: The Grader

In [ ]:
student_answer = """
Overfitting is when the model learns the training data too much.
It performs good on training data but bad on new data.
We can reduce it by using more data or regularization.
"""
grader_prompt = f"""
Grade this student answer on overfitting.

Rubric:
- Correctness: 40 marks
- Clarity: 30 marks
- Completeness: 30 marks

Return score, strengths, improvements and suggested improved answer.

Student answer:
{student_answer}
"""
print(ask_openai(grader_prompt, max_tokens=1000))

# 8️⃣ Prompt Pattern Library

In [ ]:
pd.DataFrame([
    {"Pattern": "Explainer", "Use": "Teach concepts", "Output": "Analogy + example + misconception"},
    {"Pattern": "Examiner", "Use": "Generate MCQs", "Output": "Questions + answer key"},
    {"Pattern": "Grader", "Use": "Evaluate submissions", "Output": "Score + feedback"},
    {"Pattern": "Socratic Tutor", "Use": "Guide students", "Output": "Hints and questions"},
    {"Pattern": "Research Assistant", "Use": "Summarize papers", "Output": "Key ideas + gaps"},
])

# 🧪 Lab

Create one structured-output prompt and one reusable prompt pattern for your subject.

# ✅ Summary

Structured outputs make LLMs usable in applications.